<a href="https://colab.research.google.com/github/Aiden-Ross-Dsouza/Natural-Language-Processing/blob/main/Large_Language_Models/Llama/notebooks/Llama_3_8B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Istall/ Import Libraries

In [1]:
!pip install tiktoken blobfile

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.set_default_tensor_type(torch.BFloat16Tensor)

/usr/local/lib/python3.11/dist-packages/torch/__init__.py:1144: UserWarning: torch.set_default_tensor_type() is deprecated as of PyTorch 2.1, please use torch.set_default_dtype() and torch.set_default_device() as alternatives. (Triggered internally at ../torch/csrc/tensor/python_tensor.cpp:432.)
  _C._set_default_tensor_type(t)


# Hyper-parameters

In [3]:
dim = 4096                            # model size in table 3 of the paper
ffn_dim = 14336                       # ffn dim in table 3 of the paper
n_layers = 32                         # layers in table 3 of the paper
n_heads = 32                          # attention heads in table 3 of the paper
n_kv_heads = 8                        # Under section 3.2 of the paper
vocab_size = 128256                   # From params.json file # 128k tokens
norm_eps = 1e-5                       # From params.json file
rope_theta = 500000                   # Under table 3
max_batch_size = 4                    # Depends on each individual's local machine specs
max_seq_len = 128                     # Depends on each individual's local machine specs
n_kv_head_rep = n_heads // n_kv_heads # 16 / 8 = 2
head_dim = dim // n_heads             # 4096 / 32 = 128

# RMS Norm

In [4]:
class RMSNorm(nn.Module):
  def __init__(self, dim, norm_eps):
    super().__init__()
    self.norm_eps = norm_eps
    self.weight = nn.Parameter(torch.ones(dim)) # (embed_dim) (4096)

  def _norm(self, x):
    return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.norm_eps) # x/rt(1/N * summation(x**2)) # (2, 8, 4096)

  def forward(self, x):                    # (2, 8, dim)
    out = self._norm(x.float()).type_as(x) # (batch_size, seq_len, embed_dim) (2, 8, 4096)
    return out * self.weight               # (4096) * (2, 8, 4096) -> (2, 8, 4096)

In [5]:
dummy_inp = torch.randn(2, 8, dim)
norm = RMSNorm(dim, norm_eps)
output = norm(dummy_inp)
print(output.shape)

torch.Size([2, 8, 4096])


# Rotary Embeddings

In [6]:
def precompute_freqs_cis(dim, end, theta: float = 10000.0):
  freqs = 1.0 / (theta ** (torch.arange(0, dim, 2)[: (dim // 2)].float() / dim)) # (head_dim//2,) (64,)
  t = torch.arange(end, device=freqs.device, dtype=torch.float32) # (max_seq_len*2) (256,)
  freqs = torch.outer(t, freqs) # (max_seq_len*2, head_dim//2)
  freqs_cis = torch.polar(torch.ones_like(freqs), freqs) # (max_seq_len*2, head_dim//2)
  return freqs_cis # (max_seq_len*2, head_dim//2) (256, 64)

def reshape_for_broadcast(freqs_cis, x):
  ndim = x.ndim # 4
  assert 0 <= 1 < ndim
  # print(f"freqs_cis.shape reshape_for_broadcast: {freqs_cis.shape}")
  # print(f"x.shape[1]: {x.shape[1]}, x.shape[-1]: {x.shape[-1]}")
  assert freqs_cis.shape == (x.shape[1], x.shape[-1]//2) #freqs_cis.shape = (8, 64) x.shape = (batch_size, seq_len, n_heads, head_dim) (2, 8, 16, 128)
  shape = [d if i==1 else (d//2 if i==ndim-1 else 1) for i, d  in enumerate(x.shape)] # (1, 8, 1, 64)
  # print(shape)
  return freqs_cis.view(*shape)

def apply_rotary_emb(xq, xk, freqs_cis): # xq, xk, freqs_cis = (batch_size, seq_len, num_heads, head_dim) (2, 8, 32, 128), (batch_size, seq_len, num_kv_heads, head_dim) (2, 8, 8, 128), (8, 128)
  xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2)) # xq = (batch_size, seq_len, num_heads, head_dim) (2, 8, 32, 128) -> (batch_size, seq_len, num_heads, head_dim//2, 2) (2, 8, 16, 64, 2) -> converted to complex (2, 8, 16, 128)
  # print(f"xq_.shape: {xq_.shape}") # (2, 8, 32, 64)
  xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2)) # xk = (batch_size, seq_len, num_kv_heads, head_dim) (2, 8, 8, 128) -> (batch_size, seq_len, num_kv_heads, head_dim//2) (2, 8, 8, 64, 2) -> converted to complex (2, 8, 8, 128)
  freqs_cis = reshape_for_broadcast(freqs_cis, xq) # (1, 8, 1, 64)
  # print(f"freqs_cis.shape: {freqs_cis.shape}") # (1, 8, 1, 64)
  xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3) # (2, 8, 32, 128)
  # print(f"(xq_ *freqs_cis).shape: {(xq_ *freqs_cis).shape}") # (2, 8, 32, 64)
  # print(f"(torch.view_as_real(xq_ * freqs_cis)).shape: {(torch.view_as_real(xq_ * freqs_cis)).shape}") # (2, 8, 32, 64, 2)
  # print(f"(torch.view_as_real(xq_ * freqs_cis).flatten(3)).shape: {(torch.view_as_real(xq_ * freqs_cis).flatten(3)).shape}") # (2, 8, 32, 128)
  xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3) # (2, 8, 8, 128)
  # print(f"xq_out.shape: {xq_out.shape}")
  # print(f"xk_out.shape: {xk_out.shape}")
  return xq_out.type_as(xq), xk_out.type_as(xk) # (2, 8, 32, 128), (2, 8, 8, 128)

In [7]:
dummy_inp1 = torch.randn(2, 8, n_heads, head_dim)    # (batch_size, seq_len, num_heads, head_dim) (2, 8, 32, 128)
dummy_inp2 = torch.randn(2, 8, n_kv_heads, head_dim) # (batch_size, seq_len, num_kv_heads, head_dim) (2, 8, 8, 128)

dummy_freqs_cis = precompute_freqs_cis(head_dim, max_seq_len*2, rope_theta) # (max_seq_len*2, head_dim//2) (256, 64)
dummy_freqs_cis = dummy_freqs_cis[0 : 0 + 8]                                # (8, 64)

out1, out2 = apply_rotary_emb(dummy_inp1, dummy_inp2, dummy_freqs_cis)
print(out1.shape) # (batch_size, seq_len, n_heads, head_dim) (2, 8, 32, 128)
print(out2.shape) # (batch_size, seq_len, n_kv_heads, head_dim) (2, 8, 8, 128)

torch.Size([2, 8, 32, 128])
torch.Size([2, 8, 8, 128])


# Feed Forward SwishGLU

In [8]:
class FeedForward(nn.Module):
  def __init__(self, dim, ffn_dim):
    super().__init__()
    self.w1 = nn.Linear(dim, ffn_dim, bias=False) # (embed_dim, ffn_dim) (4096, 14336)
    self.w3 = nn.Linear(dim, ffn_dim, bias=False) # (embed_dim, ffn_dim) (4096, 14336)
    self.w2 = nn.Linear(ffn_dim, dim, bias=False) # (ffn_dim, embed_dim) (14336, 4096)

  def forward(self, x):
    swishGLU = F.silu(self.w1(x)) * self.w3(x) # (batch_size, seq_len, embed_dim) (2, 8, 4096) -> (batch_size, seq_len, ffn_dim) (2, 8, 14336)
    return self.w2(swishGLU) # (batch_size, seq_len, ffn_dim) (2, 8, 14336) -> (batch_size, seq_len, embed_dim) (2, 8, 4096)

In [9]:
dummy_inp = torch.randn(2, 8, dim)
feed_forward = FeedForward(dim, ffn_dim)
output = feed_forward(dummy_inp)
del feed_forward
print(f"input: {dummy_inp.shape}, output: {output.shape}")

input: torch.Size([2, 8, 4096]), output: torch.Size([2, 8, 4096])


# Grouped Query Attention (GQA) with KV Cache

In [10]:
class Attention(nn.Module):
  def __init__(self, dim, n_heads, n_kv_heads, head_dim):
    super().__init__()
    self.wq = nn.Linear(dim, n_heads*head_dim, bias=False)    # (embed_dim, n_heads*head_dim) (4096, 32*128=4,096)
    self.wk = nn.Linear(dim, n_kv_heads*head_dim, bias=False) # (embed_dim, n_kv_heads*head_dim) (4096, 8*128=1024)
    self.wv = nn.Linear(dim, n_kv_heads*head_dim, bias=False) # (embed_dim, n_kv_heads*head_dim) (4096, 8*128=1024)
    self.wo = nn.Linear(n_heads*head_dim, dim, bias=False)    # (n_heads*head_dim, embed_dim) (32*128=4,096, 4096)
    self.cache_k = torch.zeros(
        (
            max_batch_size,
            max_seq_len,
            n_kv_heads,
            head_dim
        )
    )
    self.cache_v = torch.zeros(
        (
            max_batch_size,
            max_seq_len,
            n_kv_heads,
            head_dim
        )
    )

  def forward(self, x, start_pos, freqs_cis, mask): # x: (batch_size, seq_len, embed_dim)
    batch_size, seq_len, _ = x.shape
    xq, xk, xv = self.wq(x), self.wk(x), self.wv(x) # xq: (batch_size, seq_len, n_heads*head_dim), xk, xv: (batch_size, seq_len, n_kv_heads*head_dim)
    xq = xq.view(xq.shape[0], xq.shape[1], n_heads, head_dim) # xq: (batch_size, seq_len, n_heads, head_dim)
    xk = xk.view(xk.shape[0], xk.shape[1], n_kv_heads, head_dim) # xk: (batch_size, seq_len, n_kv_heads, head_dim)
    xv = xv.view(xv.shape[0], xv.shape[1], n_kv_heads, head_dim) # xv: (batch_size, seq_len, n_kv_heads, head_dim)

    xq, xk = apply_rotary_emb(xq, xk, freqs_cis=freqs_cis) # xq, xk: (batch_size, seq_len, n_heads, head_dim)
    self.cache_k = self.cache_k.to(xq.device)
    self.cache_v = self.cache_v.to(xq.device)
    self.cache_k[:batch_size, start_pos : start_pos+seq_len] = xk
    self.cache_v[:batch_size, start_pos : start_pos+seq_len] = xv
    xk = self.cache_k[:batch_size, :start_pos+seq_len]
    xv = self.cache_v[:batch_size, :start_pos+seq_len]
    xk = torch.repeat_interleave(xk, dim=2, repeats=n_kv_head_rep) # (batch_size, seq_len, n_kv_heads, head_dim) -> (batch_size, seq_len, n_heads, head_dim)
    xv = torch.repeat_interleave(xv, dim=2, repeats=n_kv_head_rep) # (batch_size, seq_len, n_kv_heads, head_dim) -> (batch_size, seq_len, n_heads, head_dim)

    xq = xq.transpose(1, 2) # xq: (batch_size, seq_len, n_heads, head_dim) -> (batch_size, n_heads, seq_len, head_dim)
    xk = xk.transpose(1, 2) # xk: (batch_size, seq_len, n_heads, head_dim) -> (batch_size, n_heads, seq_len, head_dim)
    xv = xv.transpose(1, 2) # xv: (batch_size, seq_len, n_heads, head_dim) -> (batch_size, n_heads, seq_len, head_dim)

    out = F.scaled_dot_product_attention(xq, xk, xv, attn_mask=mask) # (batch_size, n_heads, seq_len, head_dim) # mask: (seq_len, seq_len)
    out = out.transpose(1, 2).contiguous().view(batch_size, seq_len,-1) # (batch_size, seq_len, n_heads, head_dim) -> (batch_size, seq_len, n_heads*head_dim)

    return self.wo(out) # (batch_size, seq_len, embed_dim)

In [11]:
attention = Attention(dim, n_heads, n_kv_heads, head_dim)
dummy_inp = torch.randn(2, 8, dim)
dummy_start_pos = 0
dummy_freqs_cis = torch.randn(8, 64)
dummy_mask = torch.randn(8, 8)
output = attention(dummy_inp, dummy_start_pos, dummy_freqs_cis, dummy_mask)
del attention

print(f"input: {dummy_inp.shape}, output: {output.shape}")

input: torch.Size([2, 8, 4096]), output: torch.Size([2, 8, 4096])


# Transformer Block

In [12]:
class TransformerBlock(nn.Module):
  def __init__(self, dim, ffn_dim, n_heads, n_kv_heads, head_dim):
    super().__init__()
    self.attn = Attention(dim, n_heads, n_kv_heads, head_dim)
    self.ffn = FeedForward(dim, ffn_dim)
    self.attention_norm = RMSNorm(dim, norm_eps)
    self.ffn_norm = RMSNorm(dim, norm_eps)

  def forward(self, x, start_pos, freqs_cis, mask):
    h = x + self.attn(self.attention_norm(x), start_pos, freqs_cis, mask)
    out = h + self.ffn(self.ffn_norm(h))
    return out

In [13]:
transformer_block = TransformerBlock(dim, ffn_dim, n_heads, n_kv_heads, head_dim)
dummy_inp = torch.randn(2, 8, dim)
dummy_start_pos = 0
dummy_freqs_cis = torch.randn(8, 64)
dummy_mask = torch.randn(8, 8)
output = transformer_block(dummy_inp, dummy_start_pos, dummy_freqs_cis, dummy_mask)
del transformer_block

print(f"input: {dummy_inp.shape}, output: {output.shape}")

input: torch.Size([2, 8, 4096]), output: torch.Size([2, 8, 4096])


# Transformer

In [14]:
class Transformer(nn.Module):
  def __init__(self, dim, ffn_dim, n_layers, n_heads, n_kv_heads, head_dim):
    super().__init__()
    self.tok_embeddings = nn.Embedding(vocab_size, dim)
    self.layers  = nn.ModuleList([TransformerBlock(dim, ffn_dim, n_heads, n_kv_heads, head_dim) for _ in range(n_layers)])
    self.norm = RMSNorm(dim, norm_eps)
    self.output = nn.Linear(dim, vocab_size, bias=False)
    self.freqs_cis = precompute_freqs_cis(head_dim, max_seq_len*2, rope_theta)

  @torch.inference_mode()
  def forward(self, tokens, start_pos):
    batch_size, seq_len, _ = tokens.shape
    h = self.token_embeddings(tokens)
    self.freqs_cis = self.freqs_cis.to(tokens.device)
    freqs_cis = self.freqs_cis[start_pos : start_pos+seq_len]

    mask = None
    if seq_len>1:
      mask = torch.full((seq_len, seq_len), float('-inf'), device=tokens.device)
      mask = torch.triu(mask, diagonal=1).to(tokens.device)

    for layer in self.layers:
      h = layer(h, start_pos, freqs_cis, mask)

    h = self.norm(h)
    logits = self.output(h).float()
    return logits

In [ ]:
transformer = Transformer(dim, ffn_dim, n_layers, n_heads, n_kv_heads, head_dim)

# Use rand instead of randn bcoz randn also generates negative numbers and nn.Embedding only accepts positive numbers
dummy_tokens = torch.rand(2, 8).long()
dummy_start_pos = 0
output = transformer(dummy_tokens, dummy_start_pos)
del transformer

print(f"input: {dummy_tokens.shape}, output: {output.shape}")